# Noise in avalanche photodiodes {#sec-apd-noise}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/apd-noise.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/apd-noise.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter5/detlib.py', 'detlib.py')
else:
    sys.path.insert(0, '..')                 # detlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import detlib as dl
dl.use_style()

*Slides 99–100 (multiplied shot noise, excess avalanche noise factor F ≈ Mˣ) and 106–107 (worked example: noise in an InGaAs APD).*

## Key relations

An APD multiplies the primary photocurrent and dark current by $M$. If every primary carrier produced exactly $M$ secondaries,
the shot noise would be multiplied too, $i_n=M[2e(I_{do}+I_{pho})B]^{1/2}$, and the SNR would be unchanged (slide 99). But impact
ionisation is random: some carriers travel far before ionising, others ionise at once (slide 100). The gain itself fluctuates, and
the noise is larger by the **excess noise factor** $F=\langle M^2\rangle/\langle M\rangle^2$:

$$
i_{n\text{-APD}}=\left[2e(I_{do}+I_{pho})M^2FB\right]^{1/2},\qquad F\approx M^x .
$$

$x\approx0.3$–$0.5$ for Si, $0.7$–$1$ for Ge and InGaAs/InP (slide 100). McIntyre's theory for a uniform avalanche region,
with the avalanche started by the carrier of larger coefficient and $k$ the ratio of the smaller to the larger coefficient, gives

$$
F=kM+(1-k)\left(2-\frac1M\right),
$$

so $F\to2$ at high gain when $k=0$, and $F\approx M$ when $k=1$. The SNR of an APD receiver with a load $R_L$ is

$$
\text{SNR}=\frac{M^2I_{pho}^2}{2e(I_{do}+I_{pho})M^2FB+4k_BTB/R_L}.
$$

Gain helps against the **thermal** noise of the load and amplifier, which is not multiplied, but it adds excess noise to the
**shot** noise. There is therefore an **optimum gain**: above it, $M^2F$ grows faster than $M^2$ and the SNR falls.

## The excess noise factor

In [ ]:
#| label: fig-F
#| fig-cap: "McIntyre excess noise factor versus gain for several k, compared with the empirical F = Mˣ (dashed) for the ranges of slide 100. Si (k ≈ 0.02–0.1) stays near F ≈ 2–5 even at M = 100; InP and InGaAs/InP APDs (k_eff ≈ 0.5–0.7) follow F ≈ M^0.7–0.9."
M = np.logspace(0, 2.5, 200)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for k, c in zip((0, 0.02, 0.1, 0.3, 0.7, 1.0), dl.PALETTE):
    ax.loglog(M, dl.excess_noise_mcintyre(M, k), color=c, label=f"McIntyre k = {k}")
for x in (0.3, 0.5, 0.7, 1.0): ax.loglog(M, M**x, "--", color=dl.INK2, lw=0.8)
ax.text(250, 250**0.3, "M^0.3", fontsize=8); ax.text(250, 250**0.7, "M^0.7", fontsize=8)
ax.set(xlabel="gain M", ylabel="excess noise factor F", ylim=(1, 400)); ax.legend(fontsize=8, ncol=2); plt.show()

## Live panel: optimum gain

```{ojs}
//| echo: false
import {qe, kB, mcintyre, responsivity} from "./det_sim.js"
viewof an = Inputs.form({
  lP: Inputs.range([-10, -5], {value: -7.7, step: 0.05, label: "log₁₀ optical power P_o (W)"}),
  R1: Inputs.range([0.3, 1], {value: 0.8, step: 0.01, label: "responsivity at M = 1 (A/W)"}),
  lId: Inputs.range([-11, -6], {value: -8, step: 0.05, label: "log₁₀ unmultiplied dark current I_do (A)"}),
  lB: Inputs.range([6, 10.5], {value: 8.845, step: 0.005, label: "log₁₀ bandwidth B (Hz)"}),
  lRL: Inputs.range([1, 6], {value: 3, step: 0.05, label: "log₁₀ load R_L (Ω)"}),
  model: Inputs.radio(["F = M^x", "McIntyre k"], {value: "F = M^x", label: "excess noise"}),
  x: Inputs.range([0.2, 1.2], {value: 0.7, step: 0.01, label: "x (if F = Mˣ)"}),
  k: Inputs.range([0, 1], {value: 0.5, step: 0.01, label: "k (if McIntyre)"})
})
```

```{ojs}
//| echo: false
anOut = {
  const P = 10 ** an.lP, Ipho = an.R1 * P, Id = 10 ** an.lId, B = 10 ** an.lB, RL = 10 ** an.lRL, th = 4 * kB * 300 * B / RL;
  const F = M => an.model === "F = M^x" ? M ** an.x : mcintyre(M, an.k);
  const S = M => (M * Ipho) ** 2 / (2 * qe * (Id + Ipho) * M * M * F(M) * B + th);
  const pts = []; let best = {M: 1, s: 0};
  for (let i = 0; i <= 400; i++) { const M = 10 ** (3 * i / 400), s = S(M); pts.push({M, s: 10 * Math.log10(s)}); if (s > best.s) best = {M, s}; }
  return {pts, best, S1: S(1), shotOnly: Ipho ** 2 / (2 * qe * (Id + Ipho) * B), th, Ipho, F};
}
Plot.plot({
  width: Math.min(width, 1050), height: 300, x: {label: "gain M", type: "log"}, y: {label: "SNR (dB)", grid: true},
  marks: [Plot.line(anOut.pts, {x: "M", y: "s", stroke: "#2a78d6", strokeWidth: 2.5}),
          Plot.ruleY([10 * Math.log10(anOut.shotOnly)], {stroke: "#1baf7a", strokeDasharray: "4,3"}),
          Plot.dot([{M: anOut.best.M, s: 10 * Math.log10(anOut.best.s)}], {x: "M", y: "s", r: 6, fill: "#e34948"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">I<sub>pho</sub> = ${(anOut.Ipho * 1e9).toPrecision(3)} nA · SNR at M = 1 (pin): ${(10 * Math.log10(anOut.S1)).toFixed(1)} dB ·
<b>optimum M = ${anOut.best.M.toFixed(1)}</b> (F = ${anOut.F(anOut.best.M).toFixed(2)}): <b>SNR = ${(10 * Math.log10(anOut.best.s)).toFixed(1)} dB</b>
(gain of ${(10 * Math.log10(anOut.best.s / anOut.S1)).toFixed(1)} dB over the pin) ·
<span style="color:#1baf7a">dashed</span>: shot-noise limit without excess noise ${(10 * Math.log10(anOut.shotOnly)).toFixed(1)} dB</div>`
```

Things to try: with a 1 kΩ load the pin is deeply thermal-noise limited and the APD gains about 20 dB at $Mpprox25$. Raise $R_L$ to 1 MΩ (a slow,
high-impedance receiver): the thermal noise drops, the optimum gain falls and the APD helps much less. Switch to McIntyre with
$k=0.02$ (Si): the optimum gain is much higher and the SNR improvement larger. The APD can never beat the dashed line (the ideal,
noiseless-gain shot-noise limit) and it gets closer the smaller $k$.

## Worked example (slides 106–107), recomputed

> InGaAs APD, $x\approx0.7$, $M=10$, unmultiplied dark current 10 nA, $B=700$ MHz. (a) APD noise current per √Hz in the dark;
> (b) noise current in 700 MHz; (c) minimum optical power for SNR = 10 if $\mathcal R(M=1)=0.8$ A/W (thermal noise neglected).

In [ ]:
#| label: tbl-apdnoise
#| tbl-cap: "Slides 106–107 recomputed. Part (c) solves the quadratic in I_pho exactly."
from scipy.optimize import brentq
M, x, Id0, B, R1 = 10, 0.7, 10e-9, 700e6, 0.8
F = M**x
a = np.sqrt(2*dl.qe*Id0*M**2*F)
Ipho = brentq(lambda I: dl.snr(I, Id0, B, np.inf, M=M, F=F) - 10, 1e-12, 1e-5)
th = dl.thermal_noise(50, B)
Ipho50 = brentq(lambda I: dl.snr(I, Id0, B, 50, M=M, F=F) - 10, 1e-12, 1e-4)
pd.DataFrame([("(a) i_n,dark/√B = √(2eI_do M^(2+x))", f"{a*1e12:.2f} pA/√Hz", "1.27 pA/√Hz"),
              ("(b) i_n,dark in 700 MHz", f"{a*np.sqrt(B)*1e9:.1f} nA", "33.5 nA"),
              ("(c) primary photocurrent for SNR = 10", f"{Ipho*1e9:.1f} nA", "17.6 nA"),
              ("(c) multiplied photocurrent", f"{M*Ipho*1e9:.0f} nA", "176 nA"),
              ("(c) total noise current", f"{dl.shot_noise(Id0 + Ipho, B, M, F)*1e9:.1f} nA", "55.7 nA"),
              ("(c) optical power", f"{Ipho/R1*1e9:.1f} nW", "22 nW"),
              ("with a 50 Ω load: thermal noise current", f"{th*1e9:.0f} nA", "—"),
              ("with a 50 Ω load: power for SNR = 10", f"{Ipho50/R1*1e9:.0f} nW", "—")],
             columns=["quantity", "computed", "slide"])

The slide neglects the load. With a realistic 50 Ω front end at 700 MHz the thermal noise current (480 nA) dwarfs the multiplied
shot noise and about nine times more power (0.2 µW) is needed: the reason APDs are used in the first place is precisely the thermal
noise that the example leaves out.

In [ ]:
#| label: fig-mopt
#| fig-cap: "SNR versus gain for the slide-106 APD (P_o = 22 nW, 700 MHz) with a 50 Ω and a 1 kΩ load, for F = M^0.7 and for a Si-like k = 0.02. The optimum gain is where the multiplied shot noise catches up with the thermal noise."
fig, ax = plt.subplots(figsize=(7.5, 3.8))
Mv = np.logspace(0, 3, 400)
for RL, ls in ((50, "-"), (1e3, "--")):
    ax.semilogx(Mv, 10*np.log10(dl.apd_snr(22e-9, 0.8, Mv, 10e-9, 700e6, x=0.7, RL=RL)), ls, color=dl.PALETTE[0], label=f"x = 0.7, R_L = {RL:g} Ω")
    ax.semilogx(Mv, 10*np.log10(dl.apd_snr(22e-9, 0.8, Mv, 10e-9, 700e6, k=0.02, RL=RL)), ls, color=dl.PALETTE[1], label=f"k = 0.02, R_L = {RL:g} Ω")
    for kw, c in ((dict(x=0.7), dl.PALETTE[0]), (dict(k=0.02), dl.PALETTE[1])):
        Mo, So = dl.apd_optimum_gain(22e-9, 0.8, 10e-9, 700e6, RL, **kw); ax.plot(Mo, 10*np.log10(So), "o", color=c)
ax.set(xlabel="M", ylabel="SNR (dB)"); ax.legend(fontsize=8); plt.show()

::: {.callout-tip collapse="true"}
## Going further
- **Optimum gain analytically.** With $F=M^x$ and dark current neglected, maximise the SNR: $M_\text{opt}^{2+x}=\frac{4k_BT/R_L}{xe(I_{pho}+I_{do})}$.
  Check it against the panel. Why does the optimum *decrease* when the signal increases?
- **Gain–bandwidth product.** With $k>0$ the avalanche takes several round trips of electrons and holes to build up; at high gain the
  bandwidth falls as $f_{3dB}\approx1/(2\pi M k\tau_{av})$, a constant gain–bandwidth product (100–300 GHz for InGaAs/InAlAs APDs).
  Small $k$ is good for noise *and* speed.
- **Electron-APDs.** HgCdTe and InAs APDs have $k\approx0$ and $F\approx1$–$1.3$ up to $M=100$: the avalanche is deterministic, almost a
  noiseless amplifier. They are used for single-photon-sensitive infrared imaging.
- **Compare with an optical preamplifier** (Ch. 4): an EDFA before a pin has a noise figure of about 3 dB, i.e. an effective "$F$" of 2
  with $M=G\approx1000$ and THz of optical bandwidth (@sec-receivers).
:::

::: {.callout-note collapse="true"}
## Complements
- Kasap, sec. 5.12.3; R. J. McIntyre, *IEEE Trans. Electron Devices* ED-13, 164 (1966).
- G. P. Agrawal, *Fiber-Optic Communication Systems*, 4th ed., sec. 4.4 (APD receivers and the optimum gain).
:::

## Try it yourself

1. Derive McIntyre's limits: $F(M\to\infty)=2$ for $k=0$, $F=M$ for $k=1$. Interpret $F=2$ physically.
2. What value of $x$ in $F=M^x$ reproduces McIntyre with $k=0.5$ at $M=10$?
3. In the panel, set the load to 50 Ω and the bandwidth to 10 GHz. Is an APD with $x=0.7$ worth it? How much does an APD with
   $k=0.1$ gain instead?
4. A colleague doubles the APD gain from 10 to 20 "to double the signal". When does this improve and when does it degrade the SNR?